### Notebook used for running simple benchmark

In [ ]:
import utils as ut
import b_run_staging as b
import j_benchmarks as j
import numpy as np
import k_metric_breakdowns as k

In [ ]:
train_test_dict = ut.load_json5('train_test_dict')
static_configs = ut.load_json5('static_configs')
runtime_configs = ut.load_json5('runtime_configs')

user_counts = ut.load_data('user_counts', 'df')
user_mapping = ut.load_data('user_mapping', 'df')
user_interactions = ut.load_data('user_interactions','df')
best_configs = ut.load_json5('best_configs')
hurdle_nb_model = ut.load_json5('hurdle_nb_model')

config_dict = ut.merge_configs(static_configs, runtime_configs, hurdle_nb_model, best_configs['cluster_smoothing'])
bin_metric_dict = ut.load_json5('bin_metric_dict')
degen_mask = ut.load_data('degen_mask', 'np')

config_nt_class, config_nt, train_test_nt_class, train_test_nt, bin_metric_nt = b.converting_dicts_to_nt(config_dict, train_test_dict, bin_metric_dict)

In [ ]:
user_counts_nt = b.df_to_nt('benchmark_user_counts_nt', user_counts)
user_interactions_nt = b.df_to_nt('benchmark_user_interactions_nt', user_interactions)
user_type_groups = (user_mapping.sort('user_id')['source_user_type'].to_numpy() == 'machine')
breakdown_groups = k.get_metric_breakdown(user_counts_nt=user_counts_nt, user_type_groups=user_type_groups, train_test_dict=train_test_dict)

## Running the benchmark models in validation

In [4]:
# Getting the model parameters for our static runner
usr_means, usr_vars, usr_p = j.get_user_hurdle_params(user_counts=user_counts, 
        n_users=user_mapping.shape[0], period_start=train_test_dict['train_start'], period_end=train_test_dict['burn_in_end'], config_dict=config_dict)
usr_coarse_means, usr_coarse_vars, usr_coarse_p = b.init_grid_hurdle(user_counts=user_counts.lazy(), n_users=user_mapping.shape[0], coarse_bins_per_day= bin_metric_dict['coarse_bins_per_day'], 
            period_start=train_test_dict['train_start'], period_end=train_test_dict['burn_in_end'], bin_metric_dict=bin_metric_dict)

In [5]:
empty_breakdown_groups = np.empty((0, 0), dtype='int8')

valid_results, _ = j.run_hurdle_benchmarks(user_counts_nt=user_counts_nt, 
    user_interactions_nt=user_interactions_nt, user_means=usr_means, user_variances=usr_vars, user_p=usr_p, 
    user_hour_means=usr_coarse_means, user_hour_variances=usr_coarse_vars, user_hour_p=usr_coarse_p, 
    period_start=train_test_dict['validation_start'], period_end=train_test_dict['validation_end'], 
    breakdown_groups=empty_breakdown_groups, config_nt=config_nt, bin_metric_nt=bin_metric_nt, degen_mask=degen_mask)

In [ ]:
valid_results = j.make_hurdle_benchmark_output_rows(results=valid_results, config_dict=config_dict, test_valid='valid')
ut.store_run_results(results=valid_results, dir='benchmarks/ll_only', run_name='benchmark_hurdle_valid')

### Running the benchmark models on test

In [4]:
# Getting the model parameters for our static runner
usr_means, usr_vars, usr_p = j.get_user_hurdle_params(user_counts=user_counts, n_users=user_mapping.shape[0], 
        period_start=train_test_dict['train_start'], period_end=train_test_dict['validation_end'], config_dict=config_dict)

usr_coarse_means, usr_coarse_vars, usr_coarse_p = b.init_grid_hurdle(user_counts=user_counts.lazy(), n_users=user_mapping.shape[0], 
        coarse_bins_per_day=bin_metric_dict['coarse_bins_per_day'], period_start=train_test_dict['train_start'], period_end=train_test_dict['validation_end'], 
        bin_metric_dict=bin_metric_dict)

In [ ]:
test_config = config_dict.copy()
test_config['ll_only'] = False
test_config_nt = config_nt_class(**test_config)

_, test_calibration_results = j.run_hurdle_benchmarks(user_counts_nt=user_counts_nt, user_interactions_nt=user_interactions_nt, 
        user_means=usr_means, user_variances=usr_vars, user_p=usr_p, user_hour_means=usr_coarse_means, user_hour_variances=usr_coarse_vars, 
        user_hour_p=usr_coarse_p, period_start=train_test_dict['test_start'], period_end=train_test_dict['test_end'], 
        breakdown_groups=breakdown_groups, config_nt=test_config_nt, bin_metric_nt=bin_metric_nt, degen_mask=degen_mask)

In [ ]:
test_results = j.make_calibration_hurdle_benchmark_output_rows(calibration_results=test_calibration_results, config_dict=test_config)
ut.store_run_results(results=test_results, dir='benchmarks/calibration', run_name='benchmark_hurdle_test')